# CALM-MS — train the nnU-Net base MS-lesion segmenter (free GPU, turnkey)

One-click pipeline: **raw FLAIR (+GT) → nnU-Net v2 → probability maps → CALM-MS cohort →
multi-site conformal study**. Runs on a free Colab T4 or Kaggle P100/T4x2. The trained model
replaces the legacy thesis segmenter (lesion precision ≈ 0.22) and gives the CALM-MS conformal
layer a strong base (FLAMeS bar: voxel Dice ≈ 0.74).

**Why this unblocks the paper:** a *single self-trained segmenter* applied to several public
cohorts gives multiple acquisition-shift sites with **consistent** segmentation — the >2-site
evidence the multi-site conformal characterization still needs.

**Data reality (read first):** raw MS FLAIR + expert masks (MSLesSeg, ISBI-2015, 3D-MR-MS) are
open but semi-gated (a registration / data-use form). This notebook does NOT scrape them; you
obtain the dataset once via its landing page, drop it in a Drive folder, and the notebook does
everything else. Everything after §2 is automatic and resumable across sessions.


## 0 · GPU + session check


In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU — enable GPU runtime (Colab: Runtime>Change runtime type>T4).')
print(sys.version)


## 1 · Persistent storage (Drive on Colab / working dir on Kaggle)

nnU-Net writes GB of preprocessed data + checkpoints. Point its three env vars at PERSISTENT
storage so a timed-out session resumes with `--c` instead of restarting. On Colab we use Drive.


In [ ]:
import os, pathlib
ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    BASE = '/kaggle/working/calmms'          # add a Kaggle Dataset for cross-session persistence
else:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/calmms'   # persists across Colab sessions
os.environ['nnUNet_raw'] = f'{BASE}/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = f'{BASE}/nnUNet_preprocessed'
os.environ['nnUNet_results'] = f'{BASE}/nnUNet_results'
for k in ['nnUNet_raw','nnUNet_preprocessed','nnUNet_results']:
    pathlib.Path(os.environ[k]).mkdir(parents=True, exist_ok=True)
print('storage:', BASE)


## 2 · Install nnU-Net v2 + fetch the CALM-MS repo scripts


In [ ]:
!pip -q install nnunetv2 nibabel scipy scikit-learn
# Clone the repo ONLY for its research/ scripts (dataset_conversion, infer_to_candidates,
# and the multi-site conformal study). Replace the URL with your fork/branch if needed.
REPO = '/content/medical-imaging-viewer' if not ON_KAGGLE else '/kaggle/working/medical-imaging-viewer'
import os
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/nicolasbonilla/medical-imaging-viewer {REPO}
os.chdir(REPO); print('repo:', REPO)


## 3 · Point the notebook at your raw dataset

Obtain a dataset once (open, but a form): **MSLesSeg** (IPLab Catania / *Scientific Data* 2025,
CC-BY) is the recommended start — FLAIR + T1 + T2 + expert mask, 75 patients / 115 scans.
Alternatives with the same flow: ISBI-2015, 3D-MR-MS (Ljubljana).

Upload/copy the unzipped dataset into a Drive folder and set `RAW_ROOT` to it. The converter
understands several folder layouts; MSLesSeg ships as `P*/T*/{FLAIR,T1,T2,MASK}.nii.gz`.


In [ ]:
RAW_ROOT = f'{BASE}/datasets/MSLesSeg'   # <-- point this at your unzipped dataset
LAYOUT   = 'mslesseg'                     # mslesseg | isbi | ljubljana | msseg | generic
assert os.path.isdir(RAW_ROOT), f'Put the dataset at {RAW_ROOT} (unzipped), then re-run this cell.'
print('cases (top-level):', sorted(os.listdir(RAW_ROOT))[:6], '...')


## 4 · Convert to nnU-Net raw format (single-FLAIR = FLAMeS-style, most portable)


In [ ]:
DATASET_ID = 501; DATASET_NAME = 'MSLesionFLAIR'
!python research/nnunet/dataset_conversion.py \
  --root '{RAW_ROOT}' --layout {LAYOUT} \
  --dataset-id {DATASET_ID} --dataset-name {DATASET_NAME}
# add  --with-t1  ONLY if every case has a co-registered T1 (keeps it single-channel otherwise)


## 5 · Plan + preprocess (extracts the dataset fingerprint, derives topology)


In [ ]:
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} -c 3d_fullres --verify_dataset_integrity


## 6 · Train fold 0 (250-epoch trainer; resumes across timed-out sessions)

Free sessions are time-limited; a fold will span several. Re-running this cell auto-resumes
from the last checkpoint (nnU-Net saves every ~50 epochs to `nnUNet_results` on Drive).
250 epochs clears the 'beat legacy 0.22-precision' bar; use the default 1000-epoch trainer
later for the target-bar (Dice≈0.74) result if you have the weekly budget.


In [ ]:
import glob
ckpt = glob.glob(f"{os.environ['nnUNet_results']}/Dataset{DATASET_ID}_*/nnUNetTrainer_250epochs*/fold_0/checkpoint_latest.pth")
resume = '--c' if ckpt else ''
print('resuming' if resume else 'fresh start')
!nnUNetv2_train {DATASET_ID} 3d_fullres 0 -tr nnUNetTrainer_250epochs {resume}


## 7 · Predict WITH PROBABILITIES on a held-out set

`--save_probabilities` is REQUIRED — CALM-MS consumes the soft posterior, not the argmax mask.
Point `IMAGES_TS` at nnU-Net-format images (channel `_0000.nii.gz`) of a cohort you did NOT
train on (e.g. ISBI-2015 or a held-out MSLesSeg split) — that held-out cohort becomes a new
acquisition-shift SITE for the conformal study.


In [ ]:
IMAGES_TS = f'{BASE}/datasets/heldout_imagesTs'   # nnU-Net format: CASE_0000.nii.gz
PREDS_TS  = f'{BASE}/preds/heldout'
os.makedirs(PREDS_TS, exist_ok=True)
!nnUNetv2_predict -i '{IMAGES_TS}' -o '{PREDS_TS}' -d {DATASET_ID} -c 3d_fullres -f 0 --save_probabilities


## 8 · Bridge each prediction to a CALM-MS cohort (`_prob.nii.gz` + `_gt.nii.gz`)

Turns the nnU-Net `.npz` posterior into the same cohort format the multi-site conformal study
consumes (`data/cohorts/nnunet-<site>/CASE_prob.nii.gz` + `_gt.nii.gz`).


In [ ]:
GT_DIR = f'{BASE}/datasets/heldout_gt'            # CASE.nii.gz expert masks (same case ids)
COHORT = f'{REPO}/data/cohorts/nnunet-heldout'
os.makedirs(COHORT, exist_ok=True)
import glob
for npz in sorted(glob.glob(f'{PREDS_TS}/*.npz')):
    case = os.path.basename(npz)[:-4]
    ref  = npz[:-4] + '.nii.gz'
    gt   = f'{GT_DIR}/{case}.nii.gz'
    !python research/nnunet/infer_to_candidates.py \
      --prob '{npz}' --reference '{ref}' --case '{case}' \
      {'--gt ' + repr(gt) if os.path.exists(gt) else ''} \
      --out-cohort '{COHORT}'
print('cohort written:', COHORT, '->', len(glob.glob(COHORT+'/*_prob.nii.gz')), 'prob maps')


## 9 · Benchmark vs the FLAMeS bar (voxel Dice ≈ 0.74) + lesion F1


In [ ]:
!python research/nnunet/benchmark.py --cohort '{COHORT}' || echo 'see benchmark.py --help for args'


## 10 · Add the new nnU-Net site to the multi-site conformal study

Register the fresh cohort as a site and re-run the honest characterization. With a nnU-Net base
applied to several held-out cohorts, you finally get the **>2 consistent acquisition-shift sites**
the paper needs — same segmenter, different scanners.


In [ ]:
# The study auto-discovers data/cohorts/*; add the nnunet-heldout cohort to the SITES dict in
# scripts/calm-ms/multisite_conformal_fdr.py (seg='nnU-Net', kind='patients', null_source=True),
# delete the stale cache, and re-run:
!rm -f scripts/calm-ms/.multisite_cache_v2.npz
!python scripts/calm-ms/multisite_conformal_fdr.py


---
### Save your trained model
`nnUNet_results/` already lives on Drive (persistent). To ship the segmenter into the app,
export `checkpoint_best.pth` + `plans.json` + `dataset.json`; `infer_to_candidates.py` is the
inference bridge the backend can call. Report which folds you trained — a single fold 0 is a
valid, honestly-scoped model.
